In [3]:
# Build VRT and migrate POLARIS shards to local
import shutil
import subprocess
from pathlib import Path

# ------------------------
# CONFIG
# ------------------------
in_dir  = Path(r"D:\My Drive\GEE_exports")          # Google Drive sync folder (source)
out_dir = Path(r"C:\NCA_DATA\POLARIS_shards_local") # Local shard staging
vrt_dir = Path(r"C:\NCA_DATA\POLARIS_vrt")          # Where VRT lives

out_dir.mkdir(parents=True, exist_ok=True)
vrt_dir.mkdir(parents=True, exist_ok=True)

# Match your shard naming
tile_paths = sorted(in_dir.glob("POLARIS_NCA_26911_10m-*.tif"))

EXPECTED_SHARDS = 4
DELETE_FROM_DRIVE_AFTER_SUCCESS = False
VERIFY_COPY_SIZE = True

# ------------------------
# GDAL tool checks
# ------------------------
def require(tool: str) -> str:
    p = shutil.which(tool)
    if p is None:
        raise RuntimeError(
            f"Required tool not found on PATH: {tool}\n"
            f"Install GDAL (e.g., conda install -c conda-forge gdal)"
        )
    return p

GDALBUILDVRT = require("gdalbuildvrt")

def run(cmd):
    print(" ".join(map(str, cmd)))
    subprocess.run(cmd, check=True)

# ------------------------
# Copy helpers
# ------------------------
def safe_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

    if VERIFY_COPY_SIZE:
        s1 = src.stat().st_size
        s2 = dst.stat().st_size
        if s1 != s2:
            raise RuntimeError(
                f"Copy size mismatch:\n  src={src} ({s1} bytes)\n  dst={dst} ({s2} bytes)"
            )

def ensure_all_local(local_paths):
    missing = [p for p in local_paths if not p.exists()]
    if missing:
        msg = "\n".join(str(p) for p in missing[:10])
        raise RuntimeError(f"Local shards missing after copy:\n{msg}")

# ------------------------
# Main pipeline
# ------------------------
if not tile_paths:
    raise FileNotFoundError(f"No POLARIS shards found in {in_dir}")

print("Found shards:")
for p in tile_paths:
    print(" ", p.name)

if len(tile_paths) != EXPECTED_SHARDS:
    print(f"[warn] expected {EXPECTED_SHARDS} shards but found {len(tile_paths)}")

# 1) Copy shards from Drive -> local
local_shards = []
for src in tile_paths:
    dst = out_dir / src.name
    if not dst.exists():
        safe_copy(src, dst)
    local_shards.append(dst)

ensure_all_local(local_shards)

# 2) Build VRT referencing LOCAL shards
vrt_path = vrt_dir / "POLARIS_26911_10m.vrt"

run([
    GDALBUILDVRT,
    "-overwrite",
    "-resolution", "highest",
    str(vrt_path),
    *map(str, local_shards),
])

print(f"[vrt] built: {vrt_path}")

# 3) Only after successful VRT build, optionally delete Drive shards
if DELETE_FROM_DRIVE_AFTER_SUCCESS:
    for src in tile_paths:
        try:
            src.unlink()
        except Exception as e:
            raise RuntimeError(f"Failed to delete source shard on Drive: {src}\n{e}")

    print(f"[del] deleted {len(tile_paths)} Drive shards")

print("\nDone. Next step: inspect the VRT grid, then translate to a single mosaic if desired.")

Found shards:
  POLARIS_NCA_26911_10m-0000000000-0000000000.tif
  POLARIS_NCA_26911_10m-0000000000-0000005120.tif
  POLARIS_NCA_26911_10m-0000000000-0000010240.tif
  POLARIS_NCA_26911_10m-0000005120-0000000000.tif
  POLARIS_NCA_26911_10m-0000005120-0000005120.tif
  POLARIS_NCA_26911_10m-0000005120-0000010240.tif
[warn] expected 4 shards but found 6
C:\Users\scottfordham\AppData\Local\anaconda3\Library\bin\gdalbuildvrt.EXE -overwrite -resolution highest C:\NCA_DATA\POLARIS_vrt\POLARIS_26911_10m.vrt C:\NCA_DATA\POLARIS_shards_local\POLARIS_NCA_26911_10m-0000000000-0000000000.tif C:\NCA_DATA\POLARIS_shards_local\POLARIS_NCA_26911_10m-0000000000-0000005120.tif C:\NCA_DATA\POLARIS_shards_local\POLARIS_NCA_26911_10m-0000000000-0000010240.tif C:\NCA_DATA\POLARIS_shards_local\POLARIS_NCA_26911_10m-0000005120-0000000000.tif C:\NCA_DATA\POLARIS_shards_local\POLARIS_NCA_26911_10m-0000005120-0000005120.tif C:\NCA_DATA\POLARIS_shards_local\POLARIS_NCA_26911_10m-0000005120-0000010240.tif
[vrt] built

In [4]:
import rasterio
from pathlib import Path

vrt_path = Path(r"C:\NCA_DATA\POLARIS_vrt\POLARIS_26911_10m.vrt")

with rasterio.open(vrt_path) as ds:
    print("CRS:", ds.crs)
    print("Transform:", ds.transform)
    print("Width:", ds.width)
    print("Height:", ds.height)
    print("Band count:", ds.count)
    print("Dtype:", ds.dtypes[0])

CRS: EPSG:26911
Transform: | 10.00, 0.00, 527730.00|
| 0.00,-10.00, 4815420.00|
| 0.00, 0.00, 1.00|
Width: 10437
Height: 7859
Band count: 42
Dtype: float32


In [5]:
import shutil
import subprocess
from pathlib import Path

vrt_path = Path(r"C:\NCA_DATA\POLARIS_vrt\POLARIS_26911_10m.vrt")
out_tif  = Path(r"C:\NCA_DATA\POLARIS_vrt\POLARIS_26911_10m_mosaic_exact.tif")

xmin = 527740.0
ymax = 4815410.0
xmax = 632090.0
ymin = 4736840.0
width = 10435
height = 7857

def require(tool: str) -> str:
    p = shutil.which(tool)
    if p is None:
        raise RuntimeError(f"Required tool not found on PATH: {tool}")
    return p

GDAL_TRANSLATE = require("gdal_translate")

cmd = [
    GDAL_TRANSLATE,
    "-of", "GTiff",
    "-a_srs", "EPSG:26911",
    "-projwin", str(xmin), str(ymax), str(xmax), str(ymin),
    "-outsize", str(width), str(height),
    "-r", "nearest",
    "-co", "TILED=YES",
    "-co", "COMPRESS=DEFLATE",
    "-co", "PREDICTOR=3",
    "-co", "BIGTIFF=YES",
    str(vrt_path),
    str(out_tif),
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)

print(f"Wrote: {out_tif}")

C:\Users\scottfordham\AppData\Local\anaconda3\Library\bin\gdal_translate.EXE -of GTiff -a_srs EPSG:26911 -projwin 527740.0 4815410.0 632090.0 4736840.0 -outsize 10435 7857 -r nearest -co TILED=YES -co COMPRESS=DEFLATE -co PREDICTOR=3 -co BIGTIFF=YES C:\NCA_DATA\POLARIS_vrt\POLARIS_26911_10m.vrt C:\NCA_DATA\POLARIS_vrt\POLARIS_26911_10m_mosaic_exact.tif
Wrote: C:\NCA_DATA\POLARIS_vrt\POLARIS_26911_10m_mosaic_exact.tif
